In [8]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
    save_features_as_parquet,
)
from image_analysis_3D.featurization_utils.granularity_utils import (
    measure_3D_granularity,
)

# from granularity import measure_3D_granularity
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    start_profiling,
    stop_profiling,
)

image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [ ]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "G8-1"
    patient = "NF0014_T1"
    channel = "AGP"
    compartment = "Cell"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{root_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)
channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)

In [10]:
# read in channel mapping
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

In [11]:
start_time, start_mem = start_profiling()

In [12]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
    mask_key_name=[channel_n_compartment_mapping[compartment]],
    raw_image_key_name=[channel_n_compartment_mapping[channel]],
)

In [13]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
if in_notebook:
    verbose = True
else:
    verbose = False
if processor_type == "CPU":
    object_measurements = measure_3D_granularity(
        object_loader=object_loader,
        radius=10,  # radius of the structuring element for background removal (CellProfiler default)
        granular_spectrum_length=16,  # range of the granular spectrum
        subsample_size=0.25,  # subsample the image for faster processing
        image_sample_size=0.25,  # further subsample for background removal
        mask_threshold=0.9,  # threshold for determining mask after interpolation
        verbose=verbose,
    )
else:
    raise ValueError(
        f"Processor type {processor_type} is not supported. Use 'CPU' only."
    )
final_df = pd.DataFrame(object_measurements)
# get the mean of each value in the array
# melt the dataframe to wide format
final_df = final_df.pivot_table(
    index=["object_id"], columns=["feature"], values=["value"]
)
final_df.columns = final_df.columns.droplevel()
final_df = final_df.reset_index()
# prepend compartment and channel to column names
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Granularity",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)
final_df.insert(0, "image_set", image_set_loader.image_set_name)
save_path = save_features_as_parquet(
    parent_path=output_parent_path,
    df=final_df,
    feature_type="Granularity",
    channel=channel,
    compartment=compartment,
    cpu_or_gpu=processor_type,
)
final_df.head()

Subsampled image: (41, 1537, 1540) -> (11, 385, 385) (factor=0.25)
Background subsampled: pixels (11, 385, 385) -> back_pixels (11, 385, 385) (image_sample_size=0.25)
Background removed via tophat filter.
Image startmean: 1172.567199, Processing 15 objects, Spectrum length: 16


Scale 1 - gs: 3.0638, currentmean: 1136.642446


Total measurements: 240
Non-zero measurements: 103
Mean granularity: 4.60


feature,image_set,object_id,Cell_AGP_Granularity_1,Cell_AGP_Granularity_2,Cell_AGP_Granularity_3,Cell_AGP_Granularity_4,Cell_AGP_Granularity_5,Cell_AGP_Granularity_6,Cell_AGP_Granularity_7,Cell_AGP_Granularity_8,Cell_AGP_Granularity_9,Cell_AGP_Granularity_10,Cell_AGP_Granularity_11,Cell_AGP_Granularity_12,Cell_AGP_Granularity_13,Cell_AGP_Granularity_14,Cell_AGP_Granularity_15,Cell_AGP_Granularity_16
0,G8-1,257,3.039817,0.204291,2.288808,6.828498,0.0,5.947752,0.0,0.0,7.269804,0.0,0.0,0.0,0.0,8.031792,0.0,0.0
1,G8-1,514,3.649283,0.566367,4.020162,7.916000,0.0,6.128115,0.0,0.0,7.261661,0.0,0.0,0.0,0.0,7.703367,0.0,0.0
2,G8-1,771,4.523693,0.918497,3.226721,7.892088,0.0,6.279786,0.0,0.0,7.208979,0.0,0.0,0.0,0.0,7.617754,0.0,0.0
3,G8-1,1028,4.313240,1.107171,3.104897,7.301266,0.0,6.212656,0.0,0.0,7.223088,0.0,0.0,0.0,0.0,7.670645,0.0,0.0
4,G8-1,1285,2.222860,1.158862,4.393459,9.072397,0.0,6.404084,0.0,0.0,7.162084,0.0,0.0,0.0,0.0,7.557035,0.0,0.0


In [14]:
stop_profiling(
    start_time=start_time,
    start_mem=start_mem,
    feature_type="Granularity",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU=processor_type,
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Granularity_{processor_type}.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: G8-1
        Feature type: Granularity
        CPU/GPU: CPU
        Peak memory (tracemalloc): 6642.07 MB
        Current memory (tracemalloc): 370.42 MB
        RSS at end: 564.32 MB
        Time elapsed:
        --- 114.98 seconds ---
        --- 1.92 minutes ---
        --- 0.03 hours ---
    


True